In [10]:
import os
   
docs_directory = "../data/text_files"
raw_docs = []
filenames = []
metadatas = []
   
for filename in os.listdir(docs_directory):
        if filename.endswith(".txt"):
            path = os.path.join(docs_directory, filename)
            with open(path, "r", encoding='utf-8') as f :
                text = f.read()
                raw_docs.append(text)
                filenames.append(filename)
                metadatas.append({"source":filename})

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
   
splitter = RecursiveCharacterTextSplitter(
        chunk_size = 600,
        chunk_overlap = 80,
        separators=["\n\n", "\n", ".", " ", ""]
    )
all_chunks = splitter.create_documents(raw_docs, metadatas=metadatas)
print(f"Nombre de chunks créés : {len(all_chunks)}")
print(f"Exemple de chunk : {all_chunks[0].page_content[:200]}...")

Nombre de chunks créés : 17
Exemple de chunk : Support Ticket — Tableau de Bord Analytics  
Ticket n° 58421  
Ouvert le : 2024-02-18 — Priorité : Haute  
Utilisateur : client_internaute42  
Service impacté : analytics-service  

Description du pro...


In [12]:
# Les chunks sont déjà des Documents LangChain avec leurs métadonnées.
documents = all_chunks


In [13]:
from langchain_huggingface import HuggingFaceEmbeddings

# Modèle gratuit exécuté localement (aucune clé API nécessaire).
# Le téléchargement du modèle se fait lors de la première exécution.
emb = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)


/home/lady/Documents/insta/ia-generative/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6306.58it/s]


In [14]:
# Créer la base vectorielle Chroma et y enregistrer les chunks
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=emb,
    collection_name="support_docs",
    persist_directory="./vector_store",
)


In [16]:
query = "Comment puis-je réinitialiser mon mot de passe ?"
results = vector_store.similarity_search(query, k=3)

print("Résultats de la recherche de similarité :")
for i, result in enumerate(results):
    print(f"Résultat {i+1} : {result.page_content[:200]}... (source: {result.metadata['source']})")

Résultats de la recherche de similarité :
Résultat 1 : Analyse :
Le flux ETL "Refresh_DailyMetrics" a dépassé sa fenêtre d’exécution habituelle
(de 5 minutes à 22 minutes).  
L’origine provient d'une augmentation anormale du volume de données (x4 en 48h).... (source: ticket_analytics.txt)
Résultat 2 : Étapes :
1. Vérifier que le compte est réellement verrouillé dans l’annuaire LDAP.
2. Identifier la cause : trop de tentatives, blocage administratif, expiration.
3. Effectuer le déblocage via l’outil... (source: procedure_reset.txt)
Résultat 3 : Étape 3 — Vérification post-démarrage  
1. Vérifier que tous les services internes se relancent automatiquement :  
   - API Gateway  
   - auth-service  
   - reporting-engine  
   - analytics-servic... (source: procedure_reset.txt)
